# 09 — MLflow Experiment Tracking

Logs all experiments from both pipeline stages into MLflow:

- **Experiment 1 — `tinycnn_binary_filter`**: TinyCNN v1–v4 training runs (Stage 1)
- **Experiment 2 — `birdnet_compression`**: PTQ variants + official INT8 reference (Stage 2)

Runs are logged retrospectively from saved checkpoints and known results.

To view the UI after running this notebook:
```
mlflow ui --backend-store-uri sqlite:///outputs/mlflow.db
```
Then open http://localhost:5000

In [5]:
import os
import sys
import torch
import mlflow

sys.path.insert(0, os.path.dirname(os.getcwd()))
import config

MODELS_DIR = os.path.join(config.OUTPUTS_DIR, "models")
DB_PATH = os.path.join(config.OUTPUTS_DIR, "mlflow.db")

mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath(DB_PATH)}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow version:      {mlflow.__version__}")

MLflow tracking URI: sqlite:////Users/qian/KWF/rainforest-audio-detection/outputs/mlflow.db
MLflow version:      3.13.0


## Experiment 1 — TinyCNN binary filter (Stage 1)

Four iterative training runs. Each version added more `not_meaningful` labels via
inference-assisted labeling, then retrained from scratch on the expanded label set.

| Version | not_meaningful clips | Label sources added |
|---|---|---|
| v1 | 2,997 | acoustic scan (energy + flatness) |
| v2 | 6,497 | + model_inference_v1 |
| v3 | 11,213 | + model_inference_v2 — **production model** |
| v4 | 16,845 | + model_inference_v3 — confirms convergence |

In [6]:
mlflow.set_experiment("tinycnn_binary_filter")

tinycnn_runs = [
    {
        "version": "v1",
        "label_iteration": 1,
        "n_not_meaningful": 2997,
        "label_sources": "background_energy,background_flatness",
        "production": False,
    },
    {
        "version": "v2",
        "label_iteration": 2,
        "n_not_meaningful": 6497,
        "label_sources": "background_energy,background_flatness,model_inference_v1",
        "production": False,
    },
    {
        "version": "v3",
        "label_iteration": 3,
        "n_not_meaningful": 11213,
        "label_sources": "background_energy,background_flatness,model_inference_v1,model_inference_v2",
        "production": True,
    },
    {
        "version": "v4",
        "label_iteration": 4,
        "n_not_meaningful": 16845,
        "label_sources": "background_energy,background_flatness,model_inference_v1,model_inference_v2,model_inference_v3",
        "production": False,
    },
]

for run_meta in tinycnn_runs:
    version = run_meta["version"]
    ck_path = os.path.join(MODELS_DIR, f"tinycnn_{version}.pth")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)

    with mlflow.start_run(run_name=f"tinycnn_{version}"):
        mlflow.set_tag("production", str(run_meta["production"]))

        mlflow.log_params({
            "version": version,
            "architecture": "TinyCNN",
            "epochs": ck["epoch"],
            "batch_size": 32,
            "optimizer": "Adam",
            "lr": 1e-3,
            "loss": "BCEWithLogitsLoss",
            "label_iteration": run_meta["label_iteration"],
            "label_sources": run_meta["label_sources"],
            "n_not_meaningful": run_meta["n_not_meaningful"],
            "n_train": ck["n_train"],
            "n_val": ck["n_val"],
        })

        mlflow.log_metrics({
            "val_loss": ck["val_loss"],
            "val_acc": ck["val_acc"],
            "not_meaningful_precision": ck["not_meaningful_precision"],
            "not_meaningful_recall": ck["not_meaningful_recall"],
            "not_meaningful_f1": ck["not_meaningful_f1"],
        })

        mlflow.log_artifact(ck_path, artifact_path="model")

    print(f"  tinycnn_{version}: F1={ck['not_meaningful_f1']:.3f}  "
          f"precision={ck['not_meaningful_precision']:.3f}  "
          f"recall={ck['not_meaningful_recall']:.3f}")

print("\nTinyCNN runs logged.")

2026/06/12 12:52:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/12 12:52:57 INFO mlflow.store.db.utils: Updating database tables
2026/06/12 12:52:57 INFO mlflow.tracking.fluent: Experiment with name 'tinycnn_binary_filter' does not exist. Creating a new experiment.


  tinycnn_v1: F1=0.982  precision=0.969  recall=0.995
  tinycnn_v2: F1=0.989  precision=0.983  recall=0.996
  tinycnn_v3: F1=0.990  precision=0.980  recall=1.000
  tinycnn_v4: F1=0.983  precision=0.967  recall=1.000

TinyCNN runs logged.


## Experiment 2 — BirdNET v2.4 compression (Stage 2)

Five runs: four PTQ attempts (ours) plus the official INT8 QAT reference.
All agreement metrics are measured against the same FP32 reference (stored
`birdnet_species` labels in `labels_progress.csv`).

| Run | Size | Agreement vs FP32 | Notes |
|---|---|---|---|
| dynamic_range | 14.2 MB | wrong | INT8 weights corrupt mel filterbank |
| int8_calibrated | 14.1 MB | 0.0% (500 clips) | same root cause |
| int8x16 | 14.4 MB | wrong | INT16 activations don't fix INT8 weights |
| **fp16** (ours) | **26.0 MB** | **95.33%** | working artifact |
| int8_qat_official | 41.0 MB | 89.64% | Zenodo reference, QAT-trained |

In [7]:
mlflow.set_experiment("birdnet_compression")

FP32_MB = 51.7

birdnet_runs = [
    {
        "run_name": "dynamic_range",
        "origin": "ours",
        "method": "dynamic_range_ptq",
        "weight_dtype": "int8",
        "activation_dtype": "float32",
        "calibration_clips": 0,
        "artifact_file": "birdnet_v2.4_int8.tflite",
        "size_mb": 14.2,
        "sanity_check_passed": 0,
        "agreement_500": None,
        "agreement_108k": None,
        "status": "failed",
    },
    {
        "run_name": "int8_calibrated",
        "origin": "ours",
        "method": "full_int8_ptq",
        "weight_dtype": "int8",
        "activation_dtype": "int8",
        "calibration_clips": 100,
        "artifact_file": "birdnet_v2.4_int8_calibrated.tflite",
        "size_mb": 14.1,
        "sanity_check_passed": 0,
        "agreement_500": 0.0,
        "agreement_108k": None,
        "status": "failed",
    },
    {
        "run_name": "int8x16",
        "origin": "ours",
        "method": "int8_weights_int16_activations_ptq",
        "weight_dtype": "int8",
        "activation_dtype": "int16",
        "calibration_clips": 100,
        "artifact_file": "birdnet_v2.4_int8x16.tflite",
        "size_mb": 14.4,
        "sanity_check_passed": 0,
        "agreement_500": None,
        "agreement_108k": None,
        "status": "failed",
    },
    {
        "run_name": "fp16_ptq",
        "origin": "ours",
        "method": "fp16_ptq",
        "weight_dtype": "float16",
        "activation_dtype": "float32",
        "calibration_clips": 0,
        "artifact_file": "birdnet_v2.4_fp16.tflite",
        "size_mb": 26.0,
        "sanity_check_passed": 1,
        "agreement_500": 1.0,
        "agreement_108k": 0.9533,
        "agreement_108k_conf030": 0.9725,
        "agreement_108k_conf035": 0.9833,
        "agreement_108k_conf040": 0.9901,
        "agreement_108k_conf050": 0.9957,
        "agreement_108k_conf060": 0.9974,
        "status": "success",
    },
    {
        "run_name": "int8_qat_official",
        "origin": "zenodo_reference",
        "method": "int8_qat",
        "weight_dtype": "int8",
        "activation_dtype": "float32",
        "calibration_clips": None,
        "artifact_file": None,  # not in outputs/models — official Zenodo download
        "size_mb": 41.0,
        "sanity_check_passed": 1,
        "agreement_500": None,
        "agreement_108k": 0.8964,
        "agreement_108k_conf030": 0.9260,
        "agreement_108k_conf035": 0.9472,
        "agreement_108k_conf040": 0.9618,
        "agreement_108k_conf050": 0.9793,
        "agreement_108k_conf060": 0.9881,
        "status": "reference",
    },
]

for run_meta in birdnet_runs:
    with mlflow.start_run(run_name=run_meta["run_name"]):
        mlflow.set_tag("status", run_meta["status"])
        mlflow.set_tag("origin", run_meta["origin"])

        params = {
            "source_model": "BirdNET_v2.4_FP32_SavedModel",
            "method": run_meta["method"],
            "weight_dtype": run_meta["weight_dtype"],
            "activation_dtype": run_meta["activation_dtype"],
            "tf_version": "2.20.0",
            "target_device": "Raspberry_Pi_4",
        }
        if run_meta["calibration_clips"] is not None:
            params["calibration_clips"] = run_meta["calibration_clips"]
        mlflow.log_params(params)

        size_mb = run_meta["size_mb"]
        metrics = {
            "size_mb": size_mb,
            "size_reduction_pct": round((1 - size_mb / FP32_MB) * 100, 1),
            "sanity_check_passed": run_meta["sanity_check_passed"],
        }
        for key in ["agreement_500", "agreement_108k", "agreement_108k_conf030",
                    "agreement_108k_conf035", "agreement_108k_conf040",
                    "agreement_108k_conf050", "agreement_108k_conf060"]:
            if run_meta.get(key) is not None:
                metrics[key] = run_meta[key]
        mlflow.log_metrics(metrics)

        if run_meta["artifact_file"]:
            artifact_path = os.path.join(MODELS_DIR, run_meta["artifact_file"])
            mlflow.log_artifact(artifact_path, artifact_path="model")

    agree_str = f"{run_meta['agreement_108k']*100:.2f}%" if run_meta.get("agreement_108k") else "n/a"
    print(f"  {run_meta['run_name']:25s}: {size_mb:.1f} MB  agreement={agree_str}  [{run_meta['status']}]")

print("\nBirdNET compression runs logged.")

2026/06/12 12:53:10 INFO mlflow.tracking.fluent: Experiment with name 'birdnet_compression' does not exist. Creating a new experiment.


  dynamic_range            : 14.2 MB  agreement=n/a  [failed]
  int8_calibrated          : 14.1 MB  agreement=n/a  [failed]
  int8x16                  : 14.4 MB  agreement=n/a  [failed]
  fp16_ptq                 : 26.0 MB  agreement=95.33%  [success]
  int8_qat_official        : 41.0 MB  agreement=89.64%  [reference]

BirdNET compression runs logged.


## View results inline

In [8]:
import pandas as pd
from mlflow.tracking import MlflowClient

client = MlflowClient()

for exp_name in ["tinycnn_binary_filter", "birdnet_compression"]:
    exp = client.get_experiment_by_name(exp_name)
    runs = client.search_runs(exp.experiment_id, order_by=["attributes.start_time ASC"])
    rows = []
    for r in runs:
        row = {"run": r.info.run_name}
        row.update(r.data.metrics)
        rows.append(row)
    print(f"\n=== {exp_name} ===")
    print(pd.DataFrame(rows).to_string(index=False))


=== tinycnn_binary_filter ===
       run  val_loss  val_acc  not_meaningful_precision  not_meaningful_recall  not_meaningful_f1
tinycnn_v1    0.0005   0.9992                     0.969                  0.995              0.982
tinycnn_v2    0.0007   0.9990                     0.983                  0.996              0.989
tinycnn_v3    0.0007   0.9984                     0.980                  1.000              0.990
tinycnn_v4    0.0036   0.9961                     0.967                  1.000              0.983

=== birdnet_compression ===
              run  size_mb  size_reduction_pct  sanity_check_passed  agreement_500  agreement_108k  agreement_108k_conf030  agreement_108k_conf035  agreement_108k_conf040  agreement_108k_conf050  agreement_108k_conf060
    dynamic_range     14.2                72.5                  0.0            NaN             NaN                     NaN                     NaN                     NaN                     NaN                     NaN
  int8_calib